# El Problema de la Inacción: Agua, Salud y Pobreza en Ciudad de México
**Equipo:** Calidad de Datos (Carlo Kiliano Ferrera, José Julian Pérez, Aldo Sebastián Altamirano)  
**IIMAS UNAM 2026 - Licenciatura en Cienncia de Datos**

> *"Limpiar el agua es lo que necesita el México de hoy: la inacción es más cara."*

# Perfilado
Este notebook documenta el proceso de Perfilado de datos para ver con que empezaremos a trabajar para demostrar la relación causal entre tres Objetivos de Desarrollo Sostenible (ODS):
1. **ODS 6 (Agua y Saneamiento):** El origen físico de la toxicidad en cuerpos de agua superficiales.
2. **ODS 3 (Salud y Bienestar):** El impacto biológico reflejado en la morbilidad por enfermedades gastrointestinales.
3. **ODS 1 (Fin de la Pobreza):** La vulnerabilidad de infraestructura y la trampa económica que representan los gastos médicos para familias marginadas.

## 1.1 Instalacion Librerias y Preparacion Entorno

In [ ]:
# Instalar librerías necesarias
#!pip install pyxlsb openpyxl
#!pip install pdfplumber pandas
#!pip install chardet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Importar librerías
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pdfplumber
import chardet

In [2]:
from pathlib import Path

def get_project_root(marker='README.md'):
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError("No se encontró la raíz")

PROJECT_ROOT = get_project_root()

print(PROJECT_ROOT)

c:\Users\kilia\OneDrive\Desktop\Antigravity\ProyectoFinal_CalidadDatos


## 1.2 Análisis Dimension DataSets

In [3]:
def extraer_tabla_pdf(archivo_pdf, salida_csv="plantas_cdmx.csv"):
    todas_filas = []
    encabezado = None
    
    with pdfplumber.open(archivo_pdf) as pdf:
        for num_pag, pagina in enumerate(pdf.pages, start=1):
            print(f"Procesando página {num_pag}...")
            tabla = pagina.extract_table()
            
            if not tabla:
                print(f"  No se encontró tabla en página {num_pag}")
                continue
            
            # Primera página: capturar encabezado
            if num_pag == 1:
                encabezado = tabla[0]
                datos_pagina = tabla[1:]   # omitimos el encabezado
            else:
                # Si la primera fila es igual al encabezado, la saltamos
                if tabla and tabla[0] == encabezado:
                    datos_pagina = tabla[1:]
                else:
                    datos_pagina = tabla
            
            print(f"  Encontradas {len(datos_pagina)} filas de datos")
            todas_filas.extend(datos_pagina)
    
    if not todas_filas:
        print("No se extrajo ninguna fila.")
        return None
    
    # Crear DataFrame
    df = pd.DataFrame(todas_filas, columns=encabezado)
    
    # Eliminar fila de total (si existe)
    df = df[~df.iloc[:, 0].astype(str).str.contains("Total", case=False, na=False)]
    df = df.dropna(how="all")
    
    # Limpiar espacios en blanco
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    
    # Convertir columnas numéricas (suponiendo nombres en el PDF)
    for col in df.columns:
        if "l/s" in col or "Capacidad" in col or "Caudal" in col:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    # Guardar CSV
    df.to_csv(salida_csv, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {salida_csv}")
    print(f"Total de plantas extraídas: {len(df)}")
    return df

In [ ]:
# PDF convertir a csv
archivo_pdf = PROJECT_ROOT / "datos" / "datos_crudos" / "Inventario_Nacional_PPyPATAR_2024_CDMX.pdf"

# Ruta completa del CSV de salida (misma carpeta de datos crudos)
ruta_salida_csv = PROJECT_ROOT / "datos" / "datos_crudos" / "plantas_cdmx.csv"

# Convertir a string porque la función espera str
df_plantas = extraer_tabla_pdf(str(archivo_pdf), str(ruta_salida_csv))

Procesando página 1...
  Encontradas 18 filas de datos
Procesando página 2...
  Encontradas 18 filas de datos
Procesando página 3...
  Encontradas 10 filas de datos

✅ CSV guardado: c:\Users\kilia\OneDrive\Desktop\Antigravity\ProyectoFinal_CalidadDatos\datos\datos_crudos\plantas_cdmx.csv
Total de plantas extraídas: 45


In [6]:
import os
import pandas as pd
import chardet
from pathlib import Path

ruta = PROJECT_ROOT / 'datos' / 'datos_crudos'

# Obtener solo archivos de interés
archivos = [f for f in os.listdir(ruta) if f.endswith(('.xlsx', '.xls', '.csv', '.xlsb'))]

print(f"Procesando {len(archivos)} archivos...\n")

for archivo in archivos:
    path_completo = os.path.join(ruta, archivo)
    try:
        if archivo.endswith('.csv'):
            # --- Detectar la codificación real del archivo ---
            with open(path_completo, 'rb') as f:
                raw_data = f.read(200000)  # Leer una muestra (200KB para mejor detección)
                resultado = chardet.detect(raw_data)
                encoding_detectado = resultado['encoding']
                confianza = resultado['confidence']
            
            # Intentar leer con la codificación detectada, con fallback
            try:
                df_headers = pd.read_csv(path_completo, encoding=encoding_detectado, low_memory=False)
                print(f"📄 Archivo: {archivo}")
                print(f"🔤 Codificación usada: {encoding_detectado} (confianza: {confianza:.2%})")
            except (UnicodeDecodeError, LookupError):
                # Fallback: latin-1 nunca falla porque mapea todos los bytes 0x00-0xFF
                df_headers = pd.read_csv(path_completo, encoding='latin-1', low_memory=False)
                print(f"📄 Archivo: {archivo}")
                print(f"⚠️ Codificación detectada ({encoding_detectado}) falló → se usó latin-1 como fallback")
        
        elif archivo.endswith('.xlsb'):
            df_headers = pd.read_excel(path_completo, engine='pyxlsb')
            print(f"📄 Archivo: {archivo}")
            print(f"🔤 Codificación: no aplica (archivo binario)")
        
        else:  # .xlsx, .xls
            df_headers = pd.read_excel(path_completo)
            print(f"📄 Archivo: {archivo}")
            print(f"🔤 Codificación: no aplica (archivo binario)")
        
        # Mostrar información común
        print(f"📊 Columnas: {df_headers.columns.tolist()}")
        print(f"📄 Dimensiones: {df_headers.shape}")
        print("-" * 50)

    except Exception as e:
        print(f"❌ Error al leer {archivo}: {e}")
        print("-" * 50)


Procesando 7 archivos...

📄 Archivo: Anuario_Morbilidad_2017.csv
⚠️ Codificación detectada (iso8859-3) falló → se usó latin-1 como fallback
📊 Columnas: ['CVE_ESTADO', 'DES_ESTADO', 'CVE_DIAGNO', 'DES_DIAGNO', 'CVE_CIE10', 'ACUMULADO', 'MENORES_1', 'DE01_A_04', 'DE05_A_09', 'DE10_A_14', 'DE15_A_19', 'DE20_A_24', 'DE25_A_44', 'DE45_A_49', 'DE50_A_59', 'DE60_A_64', 'DE65_Y_MAS', 'SE_IGNORAN', 'SSA', 'IMSS_ORD', 'ISSSTE', 'OTRAS', 'IMSS_SOL', 'D_IF', 'PEMEX', 'SEDENA', 'SEDEMAR', 'ENERO', 'FEBRERO', 'MARZO', 'ABRIL', 'MAYO', 'JUNIO', 'JULIO', 'AGOSTO', 'SEPTIEMBRE', 'OCTUBRE', 'NOVIEMBRE', 'DICIEMBRE', 'PERIODO', 'MENORES_1F', 'DE01_A_04F', 'DE05_A_09F', 'DE10_A_14F', 'DE15_A_19F', 'DE20_A_24F', 'DE25_A_44F', 'DE45_A_49F', 'DE50_A_59F', 'DE60_A_64F', 'DE65_Y_MAF', 'SE_IGNORAF', 'TOTALM', 'TOTALF', 'SSAM', 'IMSS_ORDM', 'ISSSTEM', 'OTRASM', 'IMSS_SOLM', 'D_IFM', 'PEMEXM', 'SEDENAM', 'SEDEMARM', 'SSAF', 'IMSS_ORDF', 'ISSSTEF', 'OTRASF', 'IMSS_SOLF', 'D_IFF', 'PEMEXF', 'SEDENAF', 'SEDEMARF', '

## 1.3 Análisis Completitud